In [20]:
import os
import time
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.amp import GradScaler, autocast

# Config
SEQ_LEN = 8
BATCH_SIZE = 8
LR = 1e-4
EPOCHS = 20
PATIENCE = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "liveness_checkpoint_1.pth"


In [21]:
class VideoDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, transform=None):
        self.df = df
        self.seq_len = seq_len
        self.transform = transform

        # Map labels to integers once here
        self.label_map = {"real": 0, "attack": 1}

        # Group frames by video
        self.videos = df.groupby("video_id")["frame_path"].apply(list).to_dict()
        self.labels = df.groupby("video_id")["label"].first().to_dict()
        self.video_ids = list(self.videos.keys())

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        vid = self.video_ids[idx]
        frames = self.videos[vid]
        
        # Convert string label -> int
        label = self.label_map[self.labels[vid]]

        # Sample frames
        if len(frames) >= self.seq_len:
            idxs = np.linspace(0, len(frames) - 1, self.seq_len, dtype=int)
        else:
            idxs = np.random.choice(len(frames), self.seq_len, replace=True)

        selected = [frames[i] for i in idxs]
        imgs = []
        for fp in selected:
            img = self.transform(fp)
            imgs.append(img)
        imgs = torch.stack(imgs)

        return imgs, torch.tensor(label, dtype=torch.long)


In [22]:
class CNN_LSTM(nn.Module):
    def __init__(self, hidden_size=128, num_classes=2):
        super(CNN_LSTM, self).__init__()
        base = models.mobilenet_v2(weights="IMAGENET1K_V1")
        self.feature_extractor = base.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        feat_dim = base.last_channel

        self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=hidden_size,
                            num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        feats = self.feature_extractor(x)
        feats = self.pool(feats).view(B, T, -1)
        out, _ = self.lstm(feats)
        out = self.fc(out[:, -1, :])
        return out


In [23]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

# Dataset paths
DATASET_PATH = "D:\\face_video"
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
TEST_PATH = os.path.join(DATASET_PATH, "test")
CSV_PATH = os.path.join(DATASET_PATH, "real_and_fake.csv")
FRAME_OUTPUT_PATH = os.path.join(DATASET_PATH, "frames")

# Ensure the frame output directory exists
os.makedirs(FRAME_OUTPUT_PATH, exist_ok=True)

# Function to extract frames from videos
def extract_frames(video_path, output_dir, video_id, label):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    frame_paths = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_filename = f"{video_id}_frame{frame_count}.jpg"
        frame_path = os.path.join(output_dir, frame_filename)
        cv2.imwrite(frame_path, frame)
        frame_paths.append(frame_path.replace("\\", "/"))  # Ensure consistent path format
        frame_count += 1

    cap.release()
    return frame_paths

# Create the CSV file
if not os.path.exists(CSV_PATH):
    print("CSV file not found. Creating a new one...")
    data = []
    for split in ["train", "test"]:
        split_path = os.path.join(DATASET_PATH, split)
        for label in ["real_video", "attack"]:
            label_path = os.path.join(split_path, label)
            if os.path.exists(label_path):
                for video_file in tqdm(os.listdir(label_path), desc=f"Processing {label} videos in {split}"):
                    if video_file.endswith(".mp4"):
                        video_id = os.path.splitext(video_file)[0]
                        video_path = os.path.join(label_path, video_file)
                        frame_paths = extract_frames(video_path, FRAME_OUTPUT_PATH, video_id, label)
                        for frame_path in frame_paths:
                            data.append({
                                "video_id": video_id,
                                "frame_path": frame_path,
                                "label": "real" if label == "real_video" else "attack"
                            })
    df = pd.DataFrame(data)
    df.to_csv(CSV_PATH, index=False)
    print(f"CSV file created at {CSV_PATH}")
else:
    print(f"CSV file found at {CSV_PATH}")

# Load the CSV file
df = pd.read_csv(CSV_PATH)

# Print a preview of the dataset
print("Sample of the dataset:")
print(df.head())

CSV file found at D:\face_video\real_and_fake.csv
Sample of the dataset:
   video_id                         frame_path label
0         0  D:/face_video/frames/0_frame0.jpg  real
1         0  D:/face_video/frames/0_frame1.jpg  real
2         0  D:/face_video/frames/0_frame2.jpg  real
3         0  D:/face_video/frames/0_frame3.jpg  real
4         0  D:/face_video/frames/0_frame4.jpg  real


In [24]:
def train_model(model, train_loader, val_loader, optimizer, criterion, scaler):
    best_val_loss = float("inf")
    patience_counter = 0
    start_epoch = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    if os.path.exists(CHECKPOINT):
        ckpt = torch.load(CHECKPOINT)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        running_loss, correct, total = 0, 0, 0

        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            with autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        all_labels, all_preds = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        val_acc = val_correct / val_total
        val_loss /= val_total

        
        print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f}, Acc {val_acc:.4f}")
        

        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({"model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "epoch": epoch}, CHECKPOINT)
            print("Checkpoint saved.")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    return model, history


In [25]:
# Define transforms
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Split the dataset into training and testing sets
vids = df["video_id"].unique()
random.shuffle(vids)
split_idx = int(0.8 * len(vids))
train_vids, test_vids = vids[:split_idx], vids[split_idx:]

train_df = df[df["video_id"].isin(train_vids)]
test_df = df[df["video_id"].isin(test_vids)]

# Create datasets and dataloaders
from PIL import Image

train_ds = VideoDataset(train_df, transform=lambda x: transform(Image.open(x).convert("RGB")))
test_ds = VideoDataset(test_df, transform=lambda x: transform(Image.open(x).convert("RGB")))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Print dataset sizes
print(f"Train set size: {len(train_ds)}")
print(f"Test set size: {len(test_ds)}")

# Train the model
model = CNN_LSTM().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scaler = GradScaler("cuda")

train_model(model, train_loader, test_loader, optimizer, criterion, scaler)

Train set size: 64
Test set size: 16


Epoch 1/20: 100%|██████████| 8/8 [00:07<00:00,  1.06it/s]


Epoch 1: Train Loss 0.5181, Acc 0.8125 | Val Loss 0.2689, Acc 1.0000
Checkpoint saved.


Epoch 2/20: 100%|██████████| 8/8 [00:07<00:00,  1.09it/s]


Epoch 2: Train Loss 0.1432, Acc 1.0000 | Val Loss 0.1018, Acc 1.0000
Checkpoint saved.


...

Epoch 19/20: 100%|██████████| 8/8 [00:07<00:00,  1.07it/s]


Epoch 19: Train Loss 0.0026, Acc 1.0000 | Val Loss 0.0049, Acc 1.0000
Checkpoint saved.


Epoch 20/20: 100%|██████████| 8/8 [00:07<00:00,  1.03it/s]


Epoch 20: Train Loss 0.0025, Acc 1.0000 | Val Loss 0.0045, Acc 1.0000
Checkpoint saved.


(CNN_LSTM(
   (feature_extractor): Sequential(
     (0): Conv2dNormActivation(
       (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
       (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (2): ReLU6(inplace=True)
     )
     (1): InvertedResidual(
       (conv): Sequential(
         (0): Conv2dNormActivation(
           (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
           (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
           (2): ReLU6(inplace=True)
         )
         (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
         (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       )
     )
     (2): InvertedResidual(
       (conv): Sequential(
         (0): Conv2dNormActivation(
           (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
       

In [26]:
# Updated Graph Plotting Cell

import matplotlib.pyplot as plt

# Use the `history` dictionary returned by the `train_model` function
train_loss = history.get("train_loss", [])
val_loss = history.get("val_loss", [])
train_acc = history.get("train_acc", [])
val_acc = history.get("val_acc", [])


plt.figure(figsize=(12, 5))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Train Loss", marker='o')
plt.plot(val_loss, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid()

# Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Train Accuracy", marker='o')
plt.plot(val_acc, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

NameError: name 'history' is not defined

In [27]:


from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Initialize lists for true and predicted labels
y_true, y_pred = [], []

# Evaluate the model on the test dataset
model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Generate classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["real", "attack"]))

# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["real", "attack"], yticklabels=["real", "attack"])
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

Classification Report:


ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter

In [ ]:
# transform = transforms.Compose([
#     transforms.Resize((128, 128)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406],
#                          [0.229, 0.224, 0.225])
# ])

# df = pd.read_csv("D:\\face_video\\real_and_fake.csv")  # Must contain: video_id, frame_path, label
# df = df.sample(frac=1, random_state=42)

# vids = df["video_id"].unique()
# random.shuffle(vids)
# split = int(0.8 * len(vids))
# train_vids, val_vids = vids[:split], vids[split:]
# train_df = df[df["video_id"].isin(train_vids)]
# val_df = df[df["video_id"].isin(val_vids)]

# train_ds = VideoDataset(train_df, transform=transforms.Lambda(lambda x: transform(x)))
# val_ds = VideoDataset(val_df, transform=transforms.Lambda(lambda x: transform(x)))
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# model = CNN_LSTM().to(DEVICE)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=LR)
# scaler = GradScaler("cuda")

# train_model(model, train_loader, val_loader, optimizer, criterion, scaler)
